# Goal 4 — Recovery Fine-Tuning with LoRA (Nearest-Init Only)

**Objective:** Fine-tune the vocabulary-extended `nearest_init` Qwen2.5-0.5B model on Armenian text to recover from embedding surgery and establish comparable baseline benchmarks.

**Input model:** `nearest_init` variant from Goal 3

**Training corpus:** hy_clean.txt (4.48 GB, Cleaned CC-100 Armenian)

**Hardware:** Vast.ai Instance (RTX PRO 6000 WS / A100 / H100)

## 0. Setup

In [ ]:
#!pip install transformers accelerate peft bitsandbytes datasets tqdm tabulate

In [ ]:
import torch
import os
import json
import time
import math
import random
from pathlib import Path
from tqdm import tqdm

from transformers import (AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Workstation Hardware: {torch.cuda.get_device_name(0)}")
    print(f"Total Dedicated VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Configuration & Path Alignment

In [ ]:
# CONFIGURATION
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

GOAL3_DIR = str(REPO_ROOT / "models" / "grafted_tokenizers")
MODEL_DIRS = {"nearest_init": os.path.join(GOAL3_DIR, "nearest_init")}
TOKENIZER_DIR = os.path.join(GOAL3_DIR, "extended_tokenizer")

# Full cleaned corpus is intentionally not committed. Set HY_CLEAN_PATH
# to its location, or place it at data/hy_clean.txt.
CORPUS_PATH = os.environ.get("HY_CLEAN_PATH", str(REPO_ROOT / "data" / "hy_clean.txt"))
EVAL_PATH = str(REPO_ROOT / "data" / "sample" / "hy_sample_raw.txt")
OUTPUT_DIR = str(REPO_ROOT / "outputs" / "goal4_finetuned")

os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_TRAIN_LINES = 500_000
MAX_SEQ_LEN = 512
BATCH_SIZE = 4
GRAD_ACCUM = 8
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
LOGGING_STEPS = 50
SAVE_STEPS = 500
EVAL_STEPS = 500
SEED = 42

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj"]

print("File check:")
for name, path in MODEL_DIRS.items():
    print(f"  {name}: {'OK' if os.path.isdir(path) else 'MISSING'} ({path})")
print(f"  tokenizer: {'OK' if os.path.isdir(TOKENIZER_DIR) else 'MISSING'}")
print(f"  hy_clean.txt corpus: {'OK' if os.path.exists(CORPUS_PATH) else 'MISSING'}")
print(f"  hy_sample_raw.txt evaluation: {'OK' if os.path.exists(EVAL_PATH) else 'MISSING'}")


## 2. Loading Extended Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer vocabulary size: {len(tokenizer):,}")
print(f"Pad Token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

## 3. Prepare Training Data (Data Packing)

In [ ]:
def is_armenian_char(c):
    cp = ord(c)
    return (0x0530 <= cp <= 0x058F) or (0xFB13 <= cp <= 0xFB17)

def load_armenian_lines(path, max_lines, min_len=20, min_arm_ratio=0.3):
    lines = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if len(line) < min_len:
                continue
            arm = sum(1 for c in line if is_armenian_char(c))
            alpha = sum(1 for c in line if c.isalpha())
            if alpha > 0 and arm >= 5 and (arm / alpha) >= min_arm_ratio:
                lines.append(line)
            if len(lines) >= max_lines:
                break
    return lines

print(f"Streaming {MAX_TRAIN_LINES:,} lines from cleaned Armenian text file...")
t0 = time.time()
train_lines = load_armenian_lines(CORPUS_PATH, MAX_TRAIN_LINES)
print(f"Loaded {len(train_lines):,} valid lines in {time.time()-t0:.1f}s")

In [ ]:
def tokenize_and_pack(lines, tokenizer, max_seq_len, seed=42):
    random.seed(seed)
    random.shuffle(lines)

    print(f"Tokenizing {len(lines):,} lines...")
    all_ids = []
    eos_id = tokenizer.eos_token_id

    for line in tqdm(lines, desc="Tokenizing"):
        ids = tokenizer.encode(line, add_special_tokens=False)
        all_ids.extend(ids)
        all_ids.append(eos_id)  

    total_tokens = len(all_ids)
    n_chunks = total_tokens // max_seq_len
    all_ids = all_ids[:n_chunks * max_seq_len]

    chunks = [all_ids[i*max_seq_len:(i+1)*max_seq_len] for i in range(n_chunks)]
    print(f"Packed {total_tokens:,} tokens into {n_chunks:,} chunks of {max_seq_len}")
    return chunks

train_chunks = tokenize_and_pack(train_lines, tokenizer, MAX_SEQ_LEN, SEED)
train_dataset = Dataset.from_dict({"input_ids": train_chunks, "labels": train_chunks})
train_dataset.set_format("torch")

## 4. Preparing Evaluation Data

In [ ]:
eval_lines = load_armenian_lines(EVAL_PATH, max_lines=5000)
eval_chunks = tokenize_and_pack(eval_lines, tokenizer, MAX_SEQ_LEN, seed=123)
eval_subset = eval_chunks[:200]  # Standardized evaluation set slice

eval_dataset = Dataset.from_dict({"input_ids": eval_subset, "labels": eval_subset})
eval_dataset.set_format("torch")
print(f"Eval dataset bound with {len(eval_dataset)} static samples.")

## 5. LoRA Model Preparation Architecture

In [ ]:
def setup_lora_model(model_dir, tokenizer_dir):
    print(f"Loading baseline grafted model architecture from {model_dir}...")
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto")

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        bias="none")

    peft_model = get_peft_model(model, lora_config)

    # CRITICAL SECURITY FIX: Unfreezing embedding matrices and casting to float32
    # This bypasses the 'Attempting to unscale FP16 gradients' runtime failure
    for name, param in peft_model.named_parameters():
        if "embed" in name.lower() or "lm_head" in name.lower():
            param.requires_grad = True
            param.data = param.data.to(torch.float32)

    total_params = sum(p.numel() for p in peft_model.parameters())
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"  Total Weight Elements: {total_params:,}")
    print(f"  Trainable Weight Elements: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
    return peft_model

In [ ]:
def train_model(peft_model, train_dataset, eval_dataset, output_name):
    output_path = os.path.join(OUTPUT_DIR, output_name)

    training_args = TrainingArguments(
        output_dir=output_path,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        weight_decay=0.01,
        fp16=True,
        logging_steps=LOGGING_STEPS,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        
        # SAFE ROLLING CHECKPOINT STRATEGY: Prevents storage constraints from crashing the run
        save_strategy="steps",
        save_steps=EVAL_STEPS,
        save_total_limit=1,           # Prunes historic checkpoints to limit disk overhead (~1.3GB max storage penalty)
        load_best_model_at_end=False, 
        
        seed=SEED,
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=False)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator)

    print(f"\nLaunching Recovery Run Execution Loop: {output_name}")
    print(f"  Total Optimization Track Steps: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM):,}")
    
    t0 = time.time()
    train_result = trainer.train()
    elapsed_min = (time.time() - t0) / 60

    print(f"\nFine-Tuning execution run finalized in {elapsed_min:.1f} minutes.")
    
    # Explicit manual export of final parameters
    trainer.save_model(output_path)
    tokenizer.save_pretrained(output_path)
    print(f"  Adapter weights securely saved to: {output_path}")

    eval_results = trainer.evaluate()
    final_ppl = math.exp(eval_results['eval_loss'])
    print(f"  Final Step Evaluation Loss: {eval_results['eval_loss']:.4f}")
    print(f"  Final Step Evaluation Perplexity: {final_ppl:.2f}")

    return {"name": output_name,
        "train_loss": train_result.training_loss,
        "eval_loss": eval_results["eval_loss"],
        "eval_ppl": final_ppl,
        "train_time_min": elapsed_min,
        "train_steps": train_result.global_step,
        "log_history": trainer.state.log_history}

## 6. Training Pipeline Execution: `nearest_init` Only

In [ ]:
all_results = {}

# Purged 'heuristic_init' and 'mean_init' to target 'nearest_init' in isolation, add those if reproducing our results
for strategy_name in ["nearest_init"]:
    model_dir = MODEL_DIRS[strategy_name]
    if not os.path.isdir(model_dir):
        print(f"\nExecution terminated — Target directory not found: {model_dir}")
        continue

    print(f"\n{'='*70}")
    print(f"RUNNING RECOVERY ITERATION: {strategy_name}")
    print(f"{'='*70}")

    peft_model = setup_lora_model(model_dir, TOKENIZER_DIR)
    
    result = train_model(
        peft_model, 
        train_dataset, 
        eval_dataset,
        output_name=f"lora_{strategy_name}")
    
    all_results[strategy_name] = result

    # Flush memory infrastructure
    del peft_model
    torch.cuda.empty_cache()

    print(f"\nStrategy {strategy_name} iteration phase complete. Perplexity: {result['eval_ppl']:.2f}")

## 7. Results Comparison & Cross-Verification Metrics

In [ ]:
PRE_FT_PPL = {"mean_init": 216406.1, "heuristic_init": 24484.3, "nearest_init": 96967.5}
COMPLETED_POST_FT_PPL = {"heuristic_init": 8.33, "mean_init": 10.09}
COMPLETED_TRAIN_LOSS = {"heuristic_init": 2.5141, "mean_init": 2.7720}
COMPLETED_EVAL_LOSS = {"heuristic_init": 2.1198, "mean_init": 2.3117}
COMPLETED_TIME = {"heuristic_init": 277.0, "mean_init": 28.5}

print("=" * 75)
print("         FINAL CROSS-STRATEGY COMPARISON METRICS (GOAL 4)")
print("=" * 75)
print(f"{'Strategy':<18} | {'Pre-FT PPL':>11} | {'Post-FT PPL':>12} | {'Train Loss':>10} | {'Eval Loss':>9} | {'Time':>7}")
print("-" * 75)

# Static validated benchmarks for comparison
for strategy in ["heuristic_init", "mean_init"]:
    print(f"{strategy:<18} | {PRE_FT_PPL[strategy]:>11,.1f} | {COMPLETED_POST_FT_PPL[strategy]:>12.2f} | {COMPLETED_TRAIN_LOSS[strategy]:>10.4f} | {COMPLETED_EVAL_LOSS[strategy]:>9.4f} | {COMPLETED_TIME[strategy]:>5.1f}m")

if "nearest_init" in all_results:
    nr = all_results["nearest_init"]
    print(f"nearest_init       | {PRE_FT_PPL['nearest_init']:>11,.1f} | {nr['eval_ppl']:>12.2f} | {nr['train_loss']:>10.4f} | {nr['eval_loss']:>9.4f} | {nr['train_time_min']:>5.1f}m")
print("=" * 75)

## 8. Match baseline generation

In [ ]:
from peft import PeftModel
import logging
from transformers import logging as transformers_logging

# Silencing the repetitive generation token warnings
transformers_logging.set_verbosity_error()

test_prompts = ["Հայաստանը գտնվում է ", "Ամեն դեպքում պետք է ", "Հայաստանի մայրաքաղաքը ", "2017 թվականին սկսվել է "]


evaluation_matrix = {
    "Heuristic Init": {
        "base": str(REPO_ROOT / "models" / "grafted_tokenizers" / "heuristic_init"),
        "adapter": str(REPO_ROOT / "models" / "lora_adapters" / "lora_heuristic_init")
    },
    "Mean Init": {
        "base": str(REPO_ROOT / "models" / "grafted_tokenizers" / "mean_init"),
        "adapter": str(REPO_ROOT / "models" / "lora_adapters" / "lora_mean_init")},
    "Nearest Init (LIVE)": {
        "base": str(REPO_ROOT / "models" / "grafted_tokenizers" / "nearest_init"),
        "adapter": str(REPO_ROOT / "models" / "lora_adapters" / "lora_nearest_init")}}

print("=" * 70)
print("     MATRICES ALIGNED: EXECUTING SIDE-BY-SIDE EVALUATION")
print("=" * 70)

for strategy_name, paths in evaluation_matrix.items():
    if not os.path.exists(paths["base"]):
        print(f"\n[Skipping] Base weights missing for {strategy_name} at {paths['base']}")
        continue
    if not os.path.exists(paths["adapter"]):
        print(f"\n[Skipping] Trained adapter missing for {strategy_name} at {paths['adapter']}")
        continue
        
    print(f"\nInstantiating {strategy_name} (Loading matched base + adapter layer pairs)...")
    try:
        # Loading the unique base variant
        base_model = AutoModelForCausalLM.from_pretrained(
            paths["base"],
            trust_remote_code=True,
            torch_dtype=torch.float16,
            device_map="auto")
        
        # Merging its specific recovery adapter
        model_to_eval = PeftModel.from_pretrained(base_model, paths["adapter"])
        model_to_eval.eval()
        
        print(f"\nGenerations for {strategy_name}:")
        print("-" * 50)
        
        for prompt in test_prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                outputs = model_to_eval.generate(
                    **inputs,
                    max_new_tokens=60,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    repetition_penalty=1.2
                )
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            print(f"Prompt:    {prompt}")
            print(f"Generated: {generated_text}")
            print("." * 40)
            
        # Clear specific weight matrices immediately from the workstation memory cache
        del model_to_eval, base_model
        torch.cuda.empty_cache()
        time.sleep(2)
        
    except Exception as e:
        print(f"Error executing evaluation cycle for {strategy_name}: {str(e)}")
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("Comprehensive side-by-side linguistic generation verification complete.")
print("=" * 70)

## 9. Export

In [ ]:
import os
import json

global_json_path = os.path.join(OUTPUT_DIR, "goal4_global_benchmarks.json")
nearest_json_path = os.path.join(OUTPUT_DIR, "lora_nearest_init", "nearest_init_results.json")

os.makedirs(os.path.dirname(global_json_path), exist_ok=True)
os.makedirs(os.path.dirname(nearest_json_path), exist_ok=True)

# 1. Compiling the Master Cohesive Global Payload
master_results = {
    "project_meta": {
        "corpus": "hy_clean.txt",
        "tokens_processed": "~34M tokens",
        "sequence_length": 512,
        "effective_batch_size": 32
    },
    "benchmarks": {
        "heuristic_init": {
            "pre_ft_ppl": 24484.3,
            "post_ft_ppl": 8.33,
            "train_loss": 2.5141,
            "eval_loss": 2.1198,
            "runtime_min": 277.0
        },
        "mean_init": {
            "pre_ft_ppl": 216406.1,
            "post_ft_ppl": 10.09,
            "train_loss": 2.7720,
            "eval_loss": 2.3117,
            "runtime_min": 28.5
        }
    }
}

# 2. Extracting and injecting Live Run Metrics cleanly
if "nearest_init" in all_results:
    live_run = all_results["nearest_init"]
    
    # Adding to master comparison sheet
    master_results["benchmarks"]["nearest_init"] = {
        "pre_ft_ppl": 96967.5,
        "post_ft_ppl": live_run["eval_ppl"],
        "train_loss": live_run["train_loss"],
        "eval_loss": live_run["eval_loss"],
        "runtime_min": live_run["train_time_min"]
    }
    
    # Isolating independent specific tracking logs payload for nearest_init
    nearest_init_payload = {
        "strategy": "nearest_init",
        "pre_ft_perplexity": 96967.5,
        "train_loss": live_run["train_loss"],
        "eval_loss": live_run["eval_loss"],
        "eval_ppl": live_run["eval_ppl"],
        "train_time_min": live_run["train_time_min"],
        "train_steps": live_run["train_steps"],
        "log_history": [
            {k: v for k, v in entry.items() if k in ["step", "loss", "eval_loss"]}
            for entry in live_run["log_history"]
            if "loss" in entry or "eval_loss" in entry
        ]
    }
    
    # Writing isolated payload to disk
    with open(nearest_json_path, "w") as f:
        json.dump(nearest_init_payload, f, indent=2)
    print(f"Success: Individual metric payload written to: {nearest_json_path}")

else:
    print("Warning: Live execution metrics for nearest_init not found in memory history.")

# 3. Write Master array sheet to disk
with open(global_json_path, "w") as f:
    json.dump(master_results, f, indent=2)
print(f"Success: Global baseline benchmarks master file written to: {global_json_path}")

In [ ]:
# CELL 11: RECONSTRUCT COMPREHENSIVE PERFORMANCE CURVES

import os
import matplotlib.pyplot as plt

OUTPUT_DIR = "./"

# 1. Manually mapping the verified historical log entries for step milestones
# These correspond exactly to the data generated in your fine-tuning logs
history_data = {
    "heuristic_init": {
        "steps": [500, 1000, 1500, 2000],
        "train_loss": [2.6012, 2.3714, 2.2755, 2.2158],
        "eval_loss": [2.5122, 2.2891, 2.1804, 2.1265]
    },
    "mean_init": {
        "steps": [500, 1000, 1500, 2000, 2425],
        "train_loss": [2.8419, 2.5846, 2.4799, 2.4165, 2.4121],
        "eval_loss": [2.7578, 2.5008, 2.3799, 2.3228, 2.3117]
    },
    "nearest_init": {
        "steps": [500, 1000, 1500, 2000, 2425],
        "train_loss": [2.6981, 2.4461, 2.3489, 2.2875, 2.2840],
        "eval_loss": [2.6114, 2.3701, 2.2562, 2.2028, 2.1927]
    }
}

# 2. Configure plotting environment styling
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
colors = {
    "heuristic_init": "#1D9E75",  # Emerald Green
    "mean_init": "#3266AD",       # Deep Blue
    "nearest_init": "#D85A30"     # Rust Orange
}

# 3. Plot Training Loss Curves (Left Subplot)
axes[0].set_title("Training Loss", fontsize=14, pad=10)
for strategy, data in history_data.items():
    axes[0].plot(
        data["steps"], 
        data["train_loss"], 
        label=strategy, 
        color=colors[strategy], 
        linewidth=2
    )
axes[0].set_xlabel("Step", fontsize=11)
axes[0].set_ylabel("Training Loss", fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

# 4. Plot Evaluation Loss Curves with Markers (Right Subplot)
axes[1].set_title("Evaluation Loss", fontsize=14, pad=10)
for strategy, data in history_data.items():
    axes[1].plot(
        data["steps"], 
        data["eval_loss"], 
        label=strategy, 
        color=colors[strategy], 
        linewidth=2, 
        marker="o", 
        markersize=5
    )
axes[1].set_xlabel("Step", fontsize=11)
axes[1].set_ylabel("Eval Loss", fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

plt.tight_layout()


try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    save_path = os.path.join(OUTPUT_DIR, "training_curves_final.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"Success: High-resolution chart exported cleanly to: {save_path}")
except Exception as e:
    print(f"Notice: Could not save image to disk path ({str(e)}), rendering layout to screen only.")

plt.show()